# BioChannel Notebook 4 — Explainable Cell AI

**Train a simple explainable model and export feature-level reasoning context.**

This notebook is part of the **BioChannel** Kaggle hackathon project. It is designed to be run inside Kaggle Notebooks and then reused by the Streamlit dashboard.

## How to use this notebook in BioChannel

1. Attach the relevant Kaggle dataset using **Add Data** in the Kaggle notebook sidebar.
2. Run the notebook end-to-end.
3. Save processed outputs into `/kaggle/working/processed/`.
4. Copy the exported CSV/JSON files into the BioChannel Streamlit app under `data/processed/`.
5. Reuse the helper functions in the app modules: `data_loader.py`, `preprocessing.py`, `simulation.py`, `info_theory.py`, and `prediction.py`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mutual_info_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

np.random.seed(42)


In [ ]:
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
PROCESSED_DIR = WORKING_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print('Input datasets available:')
for p in INPUT_DIR.glob('*'):
    print(' -', p)

## Dataset

Recommended datasets:

- Breast Cancer Wisconsin: `uciml/breast-cancer-wisconsin-data`
- Cancer RNA-seq alternative: `andrewmvd/gene-expression-cancer-rna-seq`

Science mapping:

This module supports explainability: why a sample or cell-state looks benign/malignant, sensitive/resistant, stressed/dormant, or uncertain.

In [ ]:
csv_files = list(INPUT_DIR.rglob('*.csv'))
for f in csv_files[:20]: print(f)

In [ ]:
def load_classification_or_demo(csv_files):
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            if len(df) >= 50 and len(df.select_dtypes(include=[np.number]).columns) >= 5:
                print('Using:', f)
                return df
        except Exception:
            pass
    print('No suitable dataset found. Creating demo classification dataset.')
    n = 300
    X = np.random.normal(0,1,(n,20))
    y = (X[:,:5].sum(axis=1) + np.random.normal(0,1,n) > 0).astype(int)
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(20)])
    df['diagnosis'] = np.where(y==1, 'M', 'B')
    return df

df = load_classification_or_demo(csv_files)
df.head()

In [ ]:
# Infer target column. For Wisconsin breast cancer, target is diagnosis.
target_col = None
for candidate in ['diagnosis','label','class','target','type']:
    for c in df.columns:
        if c.lower() == candidate:
            target_col = c
            break
    if target_col: break

if target_col is None:
    # fallback: last non-numeric or last column
    non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
    target_col = non_numeric[-1] if non_numeric else df.columns[-1]

feature_df = df.drop(columns=[target_col], errors='ignore').select_dtypes(include=[np.number]).copy()
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0)
target = df[target_col].astype(str)

le = LabelEncoder()
y = le.fit_transform(target)

X_train, X_test, y_train, y_test = train_test_split(feature_df, y, test_size=0.2, random_state=42, stratify=y if len(np.unique(y)) > 1 else None)
clf = RandomForestClassifier(n_estimators=150, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))

importances = pd.DataFrame({'feature': feature_df.columns, 'importance': clf.feature_importances_}).sort_values('importance', ascending=False)
importances.to_csv(PROCESSED_DIR / 'explainable_feature_importance.csv', index=False)
importances.head(15)

In [ ]:
sample_idx = 0
sample = feature_df.iloc[[sample_idx]]
probs = clf.predict_proba(sample)[0]
classes = le.inverse_transform(np.arange(len(probs)))
explanation_context = {
    'target_column': target_col,
    'predicted_class': str(classes[np.argmax(probs)]),
    'class_probabilities': {str(c): float(p) for c,p in zip(classes, probs)},
    'top_features': importances.head(10).to_dict(orient='records'),
}
with open(PROCESSED_DIR / 'explainable_ai_context.json', 'w') as f:
    json.dump(explanation_context, f, indent=2)
explanation_context

In [ ]:
plt.figure(figsize=(7,4))
top = importances.head(10).iloc[::-1]
plt.barh(top['feature'], top['importance'])
plt.title('Top Features for Explainable Cell AI')
plt.xlabel('Importance')
plt.show()

## Streamlit integration

Use:

- `explainable_feature_importance.csv` → feature importance chart
- `explainable_ai_context.json` → Gemma prompt context

Gemma prompt should explain top features, uncertainty, predicted class/state, and what the result means biologically. Avoid medical advice; frame as educational/scientific interpretation only.